# 02 資料處理與視覺化 — 參考解答

松柏護理之家退伍軍人症 line list 練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# -- CJK font setup (避免中文標籤顯示為方框) --
plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


## 題目 1：讀入並檢視資料

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"資料維度：{df.shape[0]} 筆 × {df.shape[1]} 欄")
print(f"\n欄位名稱：{df.columns.tolist()}")
df.head()

In [ ]:
df.info()

## 題目 2：日期轉換與衍生變項

In [ ]:
# 日期轉換
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")

# 建立 infected 欄位
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 計算 onset_to_hosp_days
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# 印出前 10 位感染者
infected_df = df[df["infected"] == 1]
infected_df[["case_id", "symptom_onset_date", "onset_to_hosp_days"]].head(10)

## 題目 3：流行曲線

In [ ]:
cases = df[df["infected"] == 1]
daily = cases.groupby("symptom_onset_date").size().rename("cases")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, color="#2c7fb8", edgecolor="white")
ax.set_title("退伍軍人症流行曲線（依發病日）", fontsize=14)
ax.set_xlabel("發病日期")
ax.set_ylabel("新增病例數")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 題目 4：翼區侵襲率比較圖

In [ ]:
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("各翼區侵襲率比較")
ax.set_xlabel("翼區")
ax.set_ylabel("侵襲率 (%)")

for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 題目 5（挑戰題）：互動式分層流行曲線

In [ ]:
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    title="互動式流行曲線（依樓層分層）",
    labels={"symptom_onset_date": "發病日期", "cases": "病例數", "floor": "樓層"},
)
fig.show()

### 解讀

- **流行曲線**：病例高峰集中在數天之內，呈現典型的 **共同暴露源（point source）** 型態
- **翼區比較**：3B 翼侵襲率最高，1B 翼最低 → 暴露源可能與特定區域設施有關
- **分層曲線**：若三樓流行高峰早於一樓，可能暗示暴露源在高樓層（例如水塔供水管路）